# Backtest of the live strategy

The strategy backtest. It uses the same engine as the daily pipeline (`backtest_engine.py`), so the backtest trades exactly what the live rules would have traded.

- **Rules:** `backtest_engine.WINNER` (the live rules; the tag is printed below).
- **Sizing:** the live weights (99% invested, each weight rounded down to 0.01%), as `paper_trade.py` trades
  (`LIVE_SIZING`; False = the unscaled rule weights).
- **Period:** walk-forward from 2022-04-01 to the last cached bar; decisions at the close, fills at the next open, 0.1% cost per side
  (live decides and trades at 2:30 PM CT with limit orders at the bid/ask; the backtest does not model that).
  Fills follow the live planner: each Friday every pick is brought back to its weight when it is more than 1 point off
  (`rebalance_band`); Mon/Wed trade only swaps and exits.
  The *never-seen* part (2022-04 → 2024-09) was not used when the rules were chosen, so it is the most honest number.
- **Data:** the cached daily bars in `Reports/cache/bars_daily_long.pkl` (no API quota is used). The stock list is today's
  universe, picked with hindsight, so expect live results to be weaker than the backtest.
- **Outputs:** `Reports/backtest_summary.csv` (the tables below) and `Reports/backtest_per_stock.csv`
  (per-stock results; shown in the app's "More about" box for the selected stock).
- Run it with *Run All*, or `python run_all.py --backtests`.

## 1. Settings
`VARIANT` = the rule keys to change for the comparison in section 5 (any key of `backtest_engine.WINNER`). Examples:
`{"midweek_exit_below": None}` (no rank-30 exit), `{"max_pick_rank": None, "cap_soft": False}` (picks from any rank, hard
sector limit), `{"midweek_swap": None}` (weekly only), `{"n": 8}` (hold 8 stocks).

In [ ]:
import pandas as pd

import backtest_engine as be

REFRESH_BARS = False                   # True = download the bars again (Alpaca market data; changes the pinned test numbers)
VARIANT = {"earnings_block_days": None}   # rule keys to change for section 5 (here: no earnings rule)
LIVE_SIZING = True                     # trade the live weights (99% invested, be.live_weights); False = rule weights
SAVE = True                            # write Reports/backtest_summary.csv and Reports/backtest_per_stock.csv
pd.set_option("display.float_format", "{:.2f}".format)
print("Live rules:", be.WINNER["tag"], "-", be.WINNER["name"])

## 2. Load prices and scores

In [ ]:
inp = be.backtest_inputs(refresh=REFRESH_BARS)
dates = inp["close"].index
U = inp["universe"]                      # the list minus stocks with < 200 bars at the end of the data (not scored)
short = [s for s in be.TRADABLE if s not in U]
print(f"{len(U)} stocks scored, bars {dates[0]:%Y-%m-%d} → {dates[-1]:%Y-%m-%d}; backtest from {be.WALK_FORWARD_START}"
      + (f"; short history, not scored: {', '.join(short)}" if short else ""))

## 3. Live rules vs QQQ and SPY
Total return, yearly growth (CAGR), Sharpe (return per unit of risk; higher is better) and the worst drop (max DD) for each period.

In [ ]:
live = be.run_rules(inp, live_sizing=LIVE_SIZING)
summary = be.period_rows(be.WINNER["tag"], live["res"]["equity"])
for bench in ("QQQ", "SPY"):
    summary += be.period_rows(f"{bench} buy & hold", be.buy_and_hold(inp, bench)["equity"])
summary = pd.DataFrame(summary)
stats = be.trade_stats(live["res"], dates)
summary.loc[(summary["Strategy"] == be.WINNER["tag"]) & (summary["Period"] == "Walk-forward"), list(stats)] = list(stats.values())
print(f"Trades: {stats['Trades']} · win rate {stats['Win rate %']:.0f}% · median trade {stats['Median trade %']:+.2f}% · "
      f"median hold {stats['Median hold (sessions)']:.0f} sessions (medians, not averages)")
summary.pivot_table(index="Period", columns="Strategy", values=["Total Return %", "Sharpe", "Max DD %"], sort=False)

**Earnings rules:** no new buy when the next earnings date is within `earnings_block_days` (5) calendar days after the
decision date, and a held pick is not topped up at the Friday rebalance before those earnings. A held stock is never sold
because of earnings. The backtest can only use the earnings dates on disk (`Reports/earnings_date.csv`), which start on 2024-09-25 and miss some
stocks, so for the earnings rule this backtest is **partial**. Section 5 compares with the earnings rule switched off
(`VARIANT = {"earnings_block_days": None}`).

In [ ]:
if be.WINNER.get("earnings_block_days"):
    earn = be.load_earnings()   # only read the calendar when the rule is actually on
    have = sorted(set(earn["Symbol"]) & set(U))
    missing = sorted(set(U) - set(have))
    print(f"Earnings rule on ({be.WINNER['earnings_block_days']} days). Dates on file for {len(have)} of {len(U)} stocks, "
          f"{earn['Earnings Date'].min():%Y-%m-%d} → {earn['Earnings Date'].max():%Y-%m-%d}; before that, and for the "
          f"{len(missing)} stocks without dates, the rule cannot act. PARTIAL test.")
    print("No dates:", " ".join(missing) or "none")
else:
    print("Earnings rule off (earnings_block_days = None).")

## 4. Per stock
For each stock: how often the rules traded it and how those trades went, next to simply holding the stock over the same
period. The line below gives the median over all stocks.

In [ ]:
per_stock = be.per_stock_table(live, inp)
med = per_stock.median(numeric_only=True)
print(f"Median stock: {med['Closed trades']:.0f} closed trades, win rate {med['Win rate %']:.0f}%, median trade "
      f"{med['Median trade %']:+.2f}%, held {med['Held % of sessions']:.0f}% of sessions; buy & hold {med['Buy & hold %']:+.0f}%")
traded = per_stock[per_stock["Closed trades"] >= 3].sort_values("Median trade %")
pd.concat([traded.tail(5), traded.head(5)])[["Symbol", "Sector", "Closed trades", "Win rate %", "Median trade %",
                                             "Median hold (sessions)", "Buy & hold %"]]

## 5. Compare a rule variant
Runs the same backtest with the keys in `VARIANT` changed (section 1) and shows both side by side. Nothing in the live
rules changes; to adopt a variant, edit `WINNER` in `backtest_engine.py` and run `python run_all.py`.

In [ ]:
variant_name = "Variant: " + ", ".join(f"{k}={v}" for k, v in VARIANT.items())
variant = be.run_rules(inp, live_sizing=LIVE_SIZING, **VARIANT)
vrows = pd.DataFrame(be.period_rows(variant_name, variant["res"]["equity"]))
vstats = be.trade_stats(variant["res"], dates)
vrows.loc[vrows["Period"] == "Walk-forward", list(vstats)] = list(vstats.values())
both = pd.concat([summary[summary["Strategy"] == be.WINNER["tag"]], vrows])
both.pivot_table(index="Period", columns="Strategy", values=["Total Return %", "Sharpe", "Max DD %"], sort=False)

## 6. Save

In [ ]:
if SAVE:
    out = pd.concat([summary, vrows], ignore_index=True)
    out.to_csv(be.REPORTS_DIR / "backtest_summary.csv", index=False)
    per_stock.to_csv(be.REPORTS_DIR / "backtest_per_stock.csv", index=False)
    print("Saved Reports/backtest_summary.csv and Reports/backtest_per_stock.csv")